## NOTEBOOK 4 ## 

## Predict Employee Performance ##

**Objective:** Use the trained model to predict performance for new/existing employees. 
**Input:** Trained model ('best_model.pkl') 
** Use-Case:** Pre-hiring screening, quarterly at-risk employee monitoring.

In [4]:
# Import all the libraries
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import LabelEncoder 

In [12]:
# Load the Saved Model
import sys
import pickle
sys.path.append(r'C:\Users\Lenovo\Downloads\Employee Performance') 
from config import MODEL, RAW_DATA, CLEANED
model_path = MODEL
with open(model_path, 'rb') as f:
    model = pickle.load(f)
print('Model loaded successfully')
print(f'Type:{type(model).__name__}') 
print(f'n_estimators:{model.n_estimators}') 
# Load the reference data for encoding 
data_raw = pd.read_excel(RAW_DATA) 
categorical_cols = ['Gender','EducationBackground','MaritalStatus','EmpDepartment','EmpJobRole',
                        'BusinessTravelFrequency','OverTime','Attrition']
label_encoders = LabelEncoder() 
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    le.fit(data_raw[col])
    label_encoders[col] = le 
# Reference : model feature columns
df_processed = pd.read_csv(CLEANED)
feature_cols = [c for c in df_processed if c != 'PerformanceRating'] 
print(f'Label Encoders ready for : {categorical_cols}') 
print(f'\n Model expects {len(feature_cols)} features in this order:') 
print(feature_cols)

Model loaded successfully
Type:RandomForestClassifier
n_estimators:100
Label Encoders ready for : ['Gender', 'EducationBackground', 'MaritalStatus', 'EmpDepartment', 'EmpJobRole', 'BusinessTravelFrequency', 'OverTime', 'Attrition']

 Model expects 26 features in this order:
['Age', 'Gender', 'EducationBackground', 'MaritalStatus', 'EmpDepartment', 'EmpJobRole', 'BusinessTravelFrequency', 'DistanceFromHome', 'EmpEducationLevel', 'EmpEnvironmentSatisfaction', 'EmpHourlyRate', 'EmpJobInvolvement', 'EmpJobLevel', 'EmpJobSatisfaction', 'NumCompaniesWorked', 'OverTime', 'EmpLastSalaryHikePercent', 'EmpRelationshipSatisfaction', 'TotalWorkExperienceInYears', 'TrainingTimesLastYear', 'EmpWorkLifeBalance', 'ExperienceYearsAtThisCompany', 'ExperienceYearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'Attrition']


## Helper Function ##
why using Helper function because without helper function the notebook becomes long, repetitive, and hard to debug. If it has helper function, each step is modular, testable and reusable especially important when we build a API on top of our model in future.

In [6]:
# Define helper function 
rating_labels = {2:'Low(2)',3:'Good(3)',4:'Excellent(4)'} 
def predict_employee(employee_data: dict) -> dict:
    employee_data : dict
    df_input = pd.DataFrame([employee_data])
    # Encode categoricals
    for col in categorical_cols:
        if col in df_input.columns:
            df_input[col] = label_encoders[col].transform(df_input[col])
# Align to model's expected feature order
    df_input = df_input[feature_cols]

    prediction   = model.predict(df_input)[0]
    probabilities = model.predict_proba(df_input)[0]
    classes      = model.classes_

    return {
        'Predicted_Rating': prediction,
        'Label'           : rating_labels[prediction],
        'Probabilities'   : dict(zip([rating_labels[c] for c in classes], probabilities.round(3)))
    }

print('Helper function defined.')

Helper function defined.


In [7]:
# Sample Prediction 
new_employees = {
    'Age' : 32,
    'Gender' : 'Male',
    'EducationBackground'          : 'Life Sciences',
    'MaritalStatus'                : 'Single',
    'EmpDepartment'                : 'Development',
    'EmpJobRole'                   : 'Research Scientist',
    'BusinessTravelFrequency'      : 'Travel_Rarely',
    'DistanceFromHome'             : 10,
    'EmpEducationLevel'            : 3,
    'EmpEnvironmentSatisfaction'   : 4,     
    'EmpHourlyRate'                : 65,
    'EmpJobInvolvement'            : 3,
    'EmpJobLevel'                  : 2,
    'EmpJobSatisfaction'           : 3,
    'NumCompaniesWorked'           : 2,
    'OverTime'                     : 'No',
    'EmpLastSalaryHikePercent'     : 18,    
    'EmpRelationshipSatisfaction'  : 3,
    'TotalWorkExperienceInYears'   : 8,
    'TrainingTimesLastYear'        : 3,
    'EmpWorkLifeBalance'           : 3,
    'ExperienceYearsAtThisCompany' : 4,
    'ExperienceYearsInCurrentRole' : 2,
    'YearsSinceLastPromotion'      : 1,     
    'YearsWithCurrManager'         : 3,
    'Attrition'                    : 'No',
}
result = predict_employee(new_employees) 
print('='*50) 
print('Employee Performance Analysis') 
print('='*50)
print(f'Predicted Rating:{result['Label']}') 
print(f'\n class Probabilities') 
for label, prob in result['Probabilities'].items():
    bar='█'* int(prob*40) 
    print(f'{label:<22} {prob:.1%} {bar}') 
print('='*50) 


Employee Performance Analysis
Predicted Rating:Good(3)

 class Probabilities
Low(2)                 1.0% 
Good(3)                97.0% ██████████████████████████████████████
Excellent(4)           2.0% 


In [ ]:
# Run prediction on full test data to simulate batch HR Screening.
df_processed = pd.read_csv(CLEANED)
X_all = df_processed.drop(columns=['PerformanceRating']) 
y_all = df_processed['PerformanceRating'] 
predictions = model.predict(X_all)
probabilities = model.predict_proba(X_all) 
# Build Otuput data frame
df_output = df_processed.copy()
df_output['Predicted_Rating'] = predictions 
df_output['Prob_Low'] = probabilities[:,0].round(3)
df_output['Prob_Good'] = probabilities[:,1].round(3)
df_output['Prob_excellent'] = probabilities[:,2].round(3) 
df_output['Correct_Prediction'] = (predictions == y_all) 
print(f'Batch Predictions complete: {len(df_output)} employees') 
print(f'Overall accuracy:{df_output['Correct_Prediction'].mean()*100:.2f}%') 
print('\nPredicted rating distribution:')
print(df_output['Predicted_Rating'].value_counts().sort_index())
# Show at-risk employees (predicted Low performance)
at_risk = df_output[df_output['Predicted_Rating'] == 2].copy()
print(f'Employees predicted as LOW PERFORMANCE: {len(at_risk)}')
print(f'\nTop 10 highest-confidence at-risk employees:')
at_risk_sorted = at_risk.sort_values('Prob_Low', ascending=False).head(10)
print(at_risk_sorted[['Prob_Low', 'EmpEnvironmentSatisfaction',
                       'EmpLastSalaryHikePercent', 'YearsSinceLastPromotion']].to_string())


Batch Predictions complete: 1200 employees
Overall accuracy:98.75%

Predicted rating distribution:
Predicted_Rating
2    190
3    885
4    125
Name: count, dtype: int64
Employees predicted as LOW PERFORMANCE: 190

Top 10 highest-confidence at-risk employees:
      Prob_Low  EmpEnvironmentSatisfaction  EmpLastSalaryHikePercent  YearsSinceLastPromotion
705       0.97                           2                        12                      7.5
618       0.97                           2                        12                      7.0
842       0.95                           2                        16                      5.0
274       0.95                           2                        13                      1.0
511       0.95                           2                        11                      7.0
709       0.95                           2                        12                      1.0
1061      0.95                           1                        11               

In [10]:
# Save predictions
import sys
sys.path.append(r'C:\Users\Lenovo\Downloads\Employee Performance')
from config import PREDICTIONS
df_output.to_csv(PREDICTIONS, index=False)

## How to Use This Model for Hiring
candidate = {
    'Age' : <age>,
    'Gender': 'Male' or 'Female''EmpDepartment': 'Development' / 'Sales' / 'Finance' / ...,
    'EmpEnvironmentSatisfaction'   :1-4,   # Critical — survey candidate
    'EmpLastSalaryHikePercent': <expected hike based on offer>,
    'YearsSinceLastPromotion': <from previous employer>,
    # ... fill remaining fields
}
result = predict_employee(candidate)
print(result['Label'])

**Key Insights:**
1. `EmpLastSalaryHikePercent` — Higher hike → higher predicted performance
2. `EmpEnvironmentSatisfaction` — Score it via structured onboarding surveys
3. `YearsSinceLastPromotion` — Long gaps indicate demotivation risk

## Conclusion ##

**Notebook-4 Complete**
* Loaded best_model.pkl for batch predictions.
* Predicted performance for all 1200 employees.
* Identified 190 at risk employees (Low Performance)
* Output: Employee_predictions.csv saved